# Sitting-posture classifier: YOLO person detection → MediaPipe landmarks → Random Forest

Pipeline per frame:
1. **YOLO** detects the person (largest `person` box is used).
2. The frame is cropped to that box (with padding) and **MediaPipe Pose** runs on the crop.
3. Landmarks are mapped back to full-frame coordinates and turned into posture features.
4. A **Random Forest** classifies the posture from those features.

Expected folders next to this notebook:

```
sliced webcam/side_normal.mp4
sliced webcam/side_lean_forward.mp4
sliced webcam/side_slouch.mp4
```

For a research claim about unseen people, collect multiple participants and split by participant. The six starter clips are only a pipeline smoke test.

## 1. Install dependencies

In [12]:
%pip install opencv-python mediapipe ultralytics pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and configuration

In [13]:
from pathlib import Path
from collections import namedtuple
import warnings
import cv2
import mediapipe as mp
from ultralytics import YOLO
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import joblib

In [14]:
# In a notebook there is no __file__, so use the current working directory.
PROJECT_DIR = Path.cwd()
VIDEO_FOLDERS = {
    "webcam": PROJECT_DIR / "sliced webcam",
}
VIDEO_LABELS = {
    "side_normal.mp4": "neutral",
    "side_lean_forward.mp4": "leaning_forward",
    "side_slouch.mp4": "slouching",
}
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
MIN_VISIBILITY = 0.50
N_SPLITS = 5      # max CV folds
N_SEGMENTS = 5    # each video is cut into this many contiguous time segments; CV holds out whole segments
RANDOM_STATE = 42

# YOLO person detector (weights are downloaded automatically on first use).
YOLO_WEIGHTS = "yolov8n.pt"   # try yolov8s.pt / yolov8m.pt for better accuracy
YOLO_CONF = 0.40              # minimum person-detection confidence
CROP_PAD = 0.15               # extra margin around the person box (fraction of box size)
yolo = YOLO(YOLO_WEIGHTS)

mp_pose = mp.solutions.pose
LANDMARK_NAMES = [item.name.lower() for item in mp_pose.PoseLandmark]
FEATURES = [
    "torso_tilt_from_vertical_deg",
    "trunk_thigh_angle_deg",
    "knee_angle_deg",
    "elbow_angle_deg",
    "head_relative_to_shoulder_x",
    "head_relative_to_shoulder_y",
]

## 3. Feature helpers

In [15]:
def joint_angle(a, b, c):
    """Angle ABC in degrees, or NaN if points are invalid."""
    p = np.asarray([a, b, c], dtype=float)
    if not np.isfinite(p).all():
        return np.nan
    ba, bc = p[0] - p[1], p[2] - p[1]
    denom = np.linalg.norm(ba) * np.linalg.norm(bc)
    if denom < 1e-9:
        return np.nan
    return float(np.degrees(np.arccos(np.clip(np.dot(ba, bc) / denom, -1, 1))))

def torso_tilt(shoulder, hip):
    """Absolute shoulder-to-hip line angle relative to image vertical."""
    if not np.isfinite([*shoulder, *hip]).all():
        return np.nan
    dx, dy = shoulder[0] - hip[0], shoulder[1] - hip[1]
    if abs(dx) + abs(dy) < 1e-9:
        return np.nan
    return float(np.degrees(np.arctan2(abs(dx), abs(dy))))

def landmark_point(landmarks, side, name):
    idx = mp_pose.PoseLandmark[f"{side.upper()}_{name.upper()}"].value
    lm = landmarks[idx]
    return np.array([lm.x, lm.y], dtype=float), float(lm.visibility)

def frame_features(landmarks):
    # Pick whichever side has better average visibility in the main joints.
    joint_names = ["shoulder", "elbow", "wrist", "hip", "knee", "ankle"]
    side_scores = {}
    for side in ("left", "right"):
        side_scores[side] = np.mean([
            landmark_point(landmarks, side, n)[1] for n in joint_names
        ])
    side = max(side_scores, key=side_scores.get)

    points, vis = {}, {}
    for name in joint_names:
        points[name], vis[name] = landmark_point(landmarks, side, name)
    nose_lm = landmarks[mp_pose.PoseLandmark.NOSE.value]
    nose = np.array([nose_lm.x, nose_lm.y], dtype=float)

    torso_len = np.linalg.norm(points["shoulder"] - points["hip"])
    if torso_len > 1e-9 and np.isfinite(nose).all():
        head_dx = float((nose[0] - points["shoulder"][0]) / torso_len)
        head_dy = float((nose[1] - points["shoulder"][1]) / torso_len)
    else:
        head_dx = head_dy = np.nan

    reliable = np.mean([vis[n] for n in ("shoulder", "hip", "knee", "ankle")]) >= MIN_VISIBILITY
    return {
        "selected_side": side,
        "pose_reliable": bool(reliable),
        "left_side_visibility_mean": float(side_scores["left"]),
        "right_side_visibility_mean": float(side_scores["right"]),
        "torso_tilt_from_vertical_deg": torso_tilt(points["shoulder"], points["hip"]),
        "trunk_thigh_angle_deg": joint_angle(points["shoulder"], points["hip"], points["knee"]),
        "knee_angle_deg": joint_angle(points["hip"], points["knee"], points["ankle"]),
        "elbow_angle_deg": joint_angle(points["shoulder"], points["elbow"], points["wrist"]),
        "head_relative_to_shoulder_x": head_dx,
        "head_relative_to_shoulder_y": head_dy,
    }

## 4. YOLO person detection and crop helpers

In [16]:
LM = namedtuple("LM", ["x", "y", "z", "visibility"])

def detect_person(frame):
    """Return (padded box x1,y1,x2,y2 in pixels, confidence) for the largest person, or (None, nan)."""
    res = yolo.predict(frame, classes=[0], conf=YOLO_CONF, verbose=False)[0]
    if res.boxes is None or len(res.boxes) == 0:
        return None, np.nan
    xyxy = res.boxes.xyxy.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()
    areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    i = int(np.argmax(areas))
    x1, y1, x2, y2 = xyxy[i]
    h, w = frame.shape[:2]
    pw, ph = (x2 - x1) * CROP_PAD, (y2 - y1) * CROP_PAD
    box = (
        int(max(0, x1 - pw)), int(max(0, y1 - ph)),
        int(min(w, x2 + pw)), int(min(h, y2 + ph)),
    )
    if box[2] - box[0] < 8 or box[3] - box[1] < 8:
        return None, np.nan
    return box, float(confs[i])

def remap_landmarks(landmarks, box, frame_w, frame_h):
    """Convert crop-normalised MediaPipe landmarks to full-frame normalised coordinates."""
    x1, y1, x2, y2 = box
    cw, ch = x2 - x1, y2 - y1
    return [
        LM((x1 + lm.x * cw) / frame_w, (y1 + lm.y * ch) / frame_h, lm.z, lm.visibility)
        for lm in landmarks
    ]

## 5. Extract landmarks from videos (YOLO → crop → MediaPipe)

In [17]:
def extract_videos():
    rows, missing = [], []
    with mp_pose.Pose(
        static_image_mode=False, model_complexity=1,
        smooth_landmarks=True, min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as pose:
        for source, folder in VIDEO_FOLDERS.items():
            print(f"\nFolder: {folder}")
            if not folder.exists():
                print(f"WARNING: folder not found: {folder}")
                continue
            for filename, label in VIDEO_LABELS.items():
                path = folder / filename
                if not path.exists():
                    missing.append(str(path))
                    print(f"MISSING: {path}")
                    continue
                cap = cv2.VideoCapture(str(path))
                if not cap.isOpened():
                    print(f"ERROR: cannot open {path}")
                    cap.release()
                    continue
                fps = cap.get(cv2.CAP_PROP_FPS)
                if not np.isfinite(fps) or fps <= 0:
                    fps = 30.0
                frame_idx = detected = reliable_count = 0
                video_id = f"{source}__{Path(filename).stem}"
                print(f"Processing {video_id} as {label}")
                while True:
                    ok, frame = cap.read()
                    if not ok:
                        break
                    frame_h, frame_w = frame.shape[:2]
                    box, person_conf = detect_person(frame)
                    lm_list = None
                    if box is not None:
                        x1, y1, x2, y2 = box
                        rgb = cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2RGB)
                        result = pose.process(rgb)
                        if result.pose_landmarks:
                            lm_list = remap_landmarks(result.pose_landmarks.landmark, box, frame_w, frame_h)
                    row = {
                        "source": source, "video_file": filename,
                        "video_id": video_id, "posture_label": label,
                        "frame_index": frame_idx, "time_seconds": frame_idx / fps,
                        "fps": fps, "frame_width": frame.shape[1],
                        "frame_height": frame.shape[0],
                        "person_detected": box is not None,
                        "person_conf": person_conf,
                        "person_box_x1": box[0] if box else np.nan,
                        "person_box_y1": box[1] if box else np.nan,
                        "person_box_x2": box[2] if box else np.nan,
                        "person_box_y2": box[3] if box else np.nan,
                        "pose_detected": lm_list is not None,
                    }
                    if lm_list is not None:
                        detected += 1
                        for i, name in enumerate(LANDMARK_NAMES):
                            lm = lm_list[i]
                            row[f"{name}_x"] = float(lm.x)
                            row[f"{name}_y"] = float(lm.y)
                            row[f"{name}_z"] = float(lm.z)
                            row[f"{name}_visibility"] = float(lm.visibility)
                        row.update(frame_features(lm_list))
                        reliable_count += int(row["pose_reliable"])
                    else:
                        for name in LANDMARK_NAMES:
                            for axis in ("x", "y", "z", "visibility"):
                                row[f"{name}_{axis}"] = np.nan
                        row.update({
                            "selected_side": None, "pose_reliable": False,
                            "left_side_visibility_mean": np.nan,
                            "right_side_visibility_mean": np.nan,
                            **{f: np.nan for f in FEATURES},
                        })
                    rows.append(row)
                    frame_idx += 1
                cap.release()
                if frame_idx:
                    print(f"  {frame_idx} frames; detected {detected/frame_idx:.1%}; reliable {reliable_count/frame_idx:.1%}")
                else:
                    print("  WARNING: no frames read")
    if not rows:
        raise RuntimeError("No frames extracted. Check PROJECT_DIR, folders, filenames, and video formats.")
    df = pd.DataFrame(rows)
    output = OUTPUT_DIR / "landmarks_per_frame.csv"
    df.to_csv(output, index=False)
    print(f"\nSaved landmarks: {output}")
    if missing:
        print("\nSome files were missing; check the messages above.")
    return df

## 6. Clean the feature table

In [18]:
def make_features(landmarks_df):
    clean = landmarks_df[
        landmarks_df["pose_detected"].fillna(False)
        & landmarks_df["pose_reliable"].fillna(False)
    ].copy()
    clean = clean.dropna(subset=FEATURES, how="all")
    if clean.empty:
        raise RuntimeError("No reliable feature rows. Check camera framing and landmark visibility.")
    path = OUTPUT_DIR / "posture_features.csv"
    clean.to_csv(path, index=False)
    print(f"\nSaved features: {path}")
    print("\nUsable frames per video:")
    print(clean.groupby(["source", "video_file", "posture_label"]).size())
    return clean

## 7. Train and evaluate

In [19]:
def train_classifier(features_df):
    classes = ["neutral", "leaning_forward", "slouching"]
    features_df = features_df.copy()
    found = set(features_df["posture_label"].dropna().unique())
    if set(classes) - found:
        raise ValueError(f"Missing classes after filtering: {set(classes)-found}")

    # Only one video per class, so split each video into contiguous time segments
    # and use those as CV groups. Scores are optimistic (same person/session) and
    # only show that the pipeline works, not how it generalises to new people.
    seg = features_df.groupby("video_id")["frame_index"].transform(
        lambda s: pd.qcut(s.rank(method="first"), N_SEGMENTS, labels=False)
    )
    features_df["segment_id"] = features_df["video_id"] + "__seg" + seg.astype(int).astype(str)

    counts = features_df.groupby("posture_label")["segment_id"].nunique()
    print("\nDistinct segments per class:")
    print(counts)
    if counts.min() < 2:
        raise ValueError("Need at least 2 segments per class for grouped cross-validation.")

    X = features_df[FEATURES].replace([np.inf, -np.inf], np.nan)
    y = features_df["posture_label"].astype(str)
    groups = features_df["segment_id"].astype(str)
    cv = StratifiedGroupKFold(
        n_splits=min(N_SPLITS, int(counts.min())),
        shuffle=True, random_state=RANDOM_STATE
    )
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            random_state=RANDOM_STATE, min_samples_leaf=3
        )),
    ])

    predictions, scores = [], []
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        if set(y_train) != set(classes):
            warnings.warn(f"Fold {fold} training data does not contain all classes.")
        model.fit(X.iloc[train_idx], y_train)
        pred = model.predict(X.iloc[test_idx])
        scores.append({
            "fold": fold,
            "accuracy": accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0),
        })
        predictions.append(pd.DataFrame({
            "video_id": features_df["video_id"].iloc[test_idx].to_numpy(),
            "true_label": y_test.to_numpy(),
            "predicted_label": pred,
            "fold": fold,
        }, index=X.iloc[test_idx].index))
        print(f"Fold {fold}: accuracy={scores[-1]['accuracy']:.3f}, macro-F1={scores[-1]['macro_f1']:.3f}")

    oof = pd.concat(predictions).sort_index()
    score_df = pd.DataFrame(scores)
    score_df.to_csv(OUTPUT_DIR / "cross_validation_scores.csv", index=False)
    oof.to_csv(OUTPUT_DIR / "out_of_fold_frame_predictions.csv", index=False)
    print("\nOut-of-fold frame-level report (preliminary):")
    print(classification_report(oof["true_label"], oof["predicted_label"], labels=classes, zero_division=0))
    print("Confusion matrix; rows=true, columns=predicted; class order:", classes)
    print(confusion_matrix(oof["true_label"], oof["predicted_label"], labels=classes))

    # Majority vote to produce one prediction for each held-out video.
    video_rows = []
    for video_id, group in oof.groupby("video_id"):
        true_label = group["true_label"].mode().iloc[0]
        pred_label = group["predicted_label"].mode().iloc[0]
        video_rows.append({
            "video_id": video_id, "true_label": true_label,
            "predicted_label": pred_label, "frames": len(group),
            "correct": true_label == pred_label,
        })
    video_results = pd.DataFrame(video_rows)
    video_results.to_csv(OUTPUT_DIR / "video_level_predictions.csv", index=False)
    print("\nVideo-level results:")
    print(video_results.to_string(index=False))
    print("Video-level accuracy:", round(video_results["correct"].mean(), 3))

    # Fit final deployment model on all current data; do not use its training score as test score.
    model.fit(X, y)
    model_path = OUTPUT_DIR / "side_posture_random_forest.joblib"
    joblib.dump({"model": model, "feature_columns": FEATURES, "classes": classes}, model_path)
    print(f"\nSaved final model: {model_path}")
    return model

## 8. Run the pipeline

In [20]:
raw = extract_videos()


Folder: c:\Users\Owner\Downloads\sitting-posture-evaluation\sliced webcam
Processing webcam__side_normal as neutral
  450 frames; detected 100.0%; reliable 100.0%
Processing webcam__side_lean_forward as leaning_forward
  450 frames; detected 100.0%; reliable 100.0%
Processing webcam__side_slouch as slouching
  450 frames; detected 100.0%; reliable 100.0%

Saved landmarks: c:\Users\Owner\Downloads\sitting-posture-evaluation\outputs\landmarks_per_frame.csv


In [21]:
features = make_features(raw)


Saved features: c:\Users\Owner\Downloads\sitting-posture-evaluation\outputs\posture_features.csv

Usable frames per video:
source  video_file             posture_label  
webcam  side_lean_forward.mp4  leaning_forward    450
        side_normal.mp4        neutral            450
        side_slouch.mp4        slouching          450
dtype: int64


In [22]:
model = train_classifier(features)
print("\nFinished. Outputs are in:", OUTPUT_DIR)


Distinct segments per class:
posture_label
leaning_forward    5
neutral            5
slouching          5
Name: segment_id, dtype: int64
Fold 1: accuracy=0.996, macro-F1=0.996
Fold 2: accuracy=1.000, macro-F1=1.000
Fold 3: accuracy=0.904, macro-F1=0.902
Fold 4: accuracy=0.985, macro-F1=0.985
Fold 5: accuracy=1.000, macro-F1=1.000

Out-of-fold frame-level report (preliminary):
                 precision    recall  f1-score   support

        neutral       1.00      1.00      1.00       450
leaning_forward       1.00      0.93      0.96       450
      slouching       0.94      1.00      0.97       450

       accuracy                           0.98      1350
      macro avg       0.98      0.98      0.98      1350
   weighted avg       0.98      0.98      0.98      1350

Confusion matrix; rows=true, columns=predicted; class order: ['neutral', 'leaning_forward', 'slouching']
[[450   0   0]
 [  0 420  30]
 [  0   1 449]]

Video-level results:
                 video_id      true_label pre